# YOLO Object Detection
We will train our own weights for the YOLOv7 model. We have one class we want to detect, snowcones. There's two datasets, one with Lidar images and one with RGB images. We can train our model to either handle both or one model for each of them.

## Importing packages

In [ ]:
import os
import re
import time
import csv

In [ ]:
!pip install --user -r requirements.txt

In [ ]:
!wget -P yolov7-main https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7_training.pt

In [ ]:
# Create local links inside your project for RGB
!mkdir -p data/rgb/images/train data/rgb/images/valid data/rgb/images/test
!mkdir -p data/rgb/labels/train data/rgb/labels/valid data/rgb/labels/test

!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/train/* data/rgb/images/train/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/valid/* data/rgb/images/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/test/* data/rgb/images/test/

!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/train/* data/rgb/labels/train/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/valid/* data/rgb/labels/valid/
#!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/test/* data/rgb/labels/test/ #There are no labels for test dataset

# Similarly for LIDAR
!mkdir -p data/lidar/combined_color/train data/lidar/combined_color/valid data/lidar/combined_color/test
!mkdir -p data/lidar/labels/train data/lidar/labels/valid data/lidar/labels/test

!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/train/* data/lidar/combined_color/train/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/valid/* data/lidar/combined_color/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/test/* data/lidar/combined_color/test/

!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/train/* data/lidar/labels/train/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/valid/* data/lidar/labels/valid/
#!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/test/* data/lidar/labels/test/ #There are no labels for test dataset

## Training YOLO model
Finding latest weights

In [ ]:

def latestWeights():
    # Get all exp folders in runs/train
    exp_dirs = [d for d in os.listdir("runs/train") if re.match(r"exp\d*$", d)]

    # Sort by numeric suffix (exp, exp2, exp3, ...)
    exp_dirs.sort(key=lambda x: int(x[3:]) if x != "exp" else 0)

    # Get the latest one
    latest_exp = exp_dirs[-1]

    best_path = f"runs/train/{latest_exp}/weights/best.pt"
    if not os.path.exists(best_path):
        print(f"Warning: best.pt not found in {latest_exp}. Using last.pt instead.")
        best_path = f"runs/train/{latest_exp}/weights/last.pt"
    print("Using weights from:", best_path)
    return best_path

Training the data, either on the latest weights or the initial COCO weights

In [ ]:
path_weights = "yolov7_training.pt"
#False -> Use COCO weights - Initilization
#True  -> Use latest weights - Continue training our model
base_training = True 
if(not base_training):
    path_weights = latestWeights()


batch_number = 32
epochs_number = 100

start_time = time.time()
!python train.py --batch {batch_number} --epochs {epochs_number} --data lidar.yaml --weights {path_weights} --device 0
path_weights = latestWeights()
!python train.py --batch {batch_number} --epochs {epochs_number} --data rgb.yaml --weights {path_weights} --device 0 

end_time = time.time()
elapsed_time = end_time - start_time
print(f"Training time: {elapsed_time:.2f} seconds") 
# Save training data time to csv file
with open("training_times.csv", "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([time.strftime("%Y-%m-%d %H:%M:%S"), elapsed_time, batch_number, epochs_number, path_weights])

## Testing weights on the test-dataset
First we find the latest weights by looking for the last exp in the 'runs/train' folder

In [ ]:
best_path = latestWeights()

Then we use these weights to do a detection test on the test dataset

In [ ]:
manual_detection = False #Change this to True if you want to select your own weights, make sure it has a "best.pt"
weight_number = 16 # expXX

if(not manual_detection):
    !python detect.py --weights {best_path} --conf 0.1 --source data/lidar/combined_color/test
    !python detect.py --weights {best_path} --conf 0.1 --source data/rgb/images/test
else:
    !python detect.py --weights runs/train/exp{weights_number}/weights/best.pt --conf 0.1 --source data/lidar/combined_color/test
    !python detect.py --weights runs/train/exp{weights_number}/weights/best.pt --conf 0.1 --source data/rgb/images/test

## Plotting Results